In [1]:
import os
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import load_model

In [2]:
smile_folder = "smile"     
non_smile_folder = "non_smile"

data = []

# Loop through the smile folder and assign label 0
if os.path.exists(smile_folder):
    for filename in os.listdir(smile_folder):
        if filename.lower().endswith(('.jpg')):
            file_path = os.path.join(smile_folder, filename)
            data.append({"image_path": file_path, "label": 0})

# Loop through the non-smile folder and assign label 1
if os.path.exists(non_smile_folder):
    for filename in os.listdir(non_smile_folder):
        if filename.lower().endswith(('.jpg')):
            file_path = os.path.join(non_smile_folder, filename)
            data.append({"image_path": file_path, "label": 1})

df = pd.DataFrame(data)

# Save the labeled dataset to a CSV file
df.to_csv("labeled_faces.csv", index=False)
print(f"Dataset successfully labeled! Total images processed: {len(df)}")

Dataset successfully labeled! Total images processed: 1202


In [3]:
IMG_SIZE = (128, 128)  
BATCH_SIZE = 32
EPOCHS = 10
CSV_FILE = "labeled_faces.csv"

# 1. LOAD AND PREPROCESS DATA
print("Loading dataset metadata...")
df = pd.read_csv(CSV_FILE)

# Helper function to load actual image arrays from paths
def load_and_preprocess_images(image_paths):
    images = []
    for path in image_paths:
        try:
            img = load_img(path, target_size=IMG_SIZE)
            img_array = img_to_array(img) / 255.0  # Normalize pixels to [0, 1]
            images.append(img_array)
        except Exception as e:
            print(f"Skipping corrupt or missing file: {path}")
    return np.array(images)

# Split metadata into train and validation sets (80% train, 20% validation)
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)

print("Processing training images...")
X_train = load_and_preprocess_images(train_df['image_path'].values)
y_train = train_df['label'].values

print("Processing validation images...")
X_val = load_and_preprocess_images(val_df['image_path'].values)
y_val = val_df['label'].values

# 2. DEFINE THE CNN MODEL ARCHITECTURE
# 2. DEFINE THE CNN MODEL ARCHITECTURE
model = models.Sequential([
    # Modern Keras standard: Define the input shape explicitly
    layers.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3)),
    
    # Convolutional Blocks to extract facial features
    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    
    # Flatten the 2D features into a 1D vector (Matches output to rank 1)
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),  # Prevents overfitting
    
    # CRITICAL FIX: Output exactly 1 probability value for binary classification
    layers.Dense(1, activation='sigmoid')
])

# Compile the model
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Verify the changes
model.summary()


# Compile the model
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# 3. TRAIN THE MODEL
print("Starting model training...")
history = model.fit(
    X_train, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_val, y_val)
)

# Save the trained weights to disk
model.save("smile_detector_model.h5")
print("Model saved as smile_detector_model.h5!")

Loading dataset metadata...
Processing training images...
Processing validation images...


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                      │ (None, 126, 126, 32)        │             896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d (MaxPooling2D)         │ (None, 63, 63, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_1 (Conv2D)                    │ (None, 61, 61, 64)          │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_1 (MaxPooling2D)       │ (None, 30, 30, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_2 (Conv2D)                    │ (None, 28, 28, 128)         │          73,856 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_2 (MaxPooling2D)       │ (None, 14, 14, 128)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten (Flatten)                    │ (None, 25088)               │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 128)                 │       3,211,392 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 1)                   │             129 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 3,304,769 (12.61 MB)

 Trainable params: 3,304,769 (12.61 MB)

 Non-trainable params: 0 (0.00 B)

Starting model training...
Epoch 1/10
31/31 ━━━━━━━━━━━━━━━━━━━━ 6s 165ms/step - accuracy: 0.5734 - loss: 0.7091 - val_accuracy: 0.8631 - val_loss: 0.6162
Epoch 2/10
31/31 ━━━━━━━━━━━━━━━━━━━━ 5s 161ms/step - accuracy: 0.7617 - loss: 0.5514 - val_accuracy: 0.8838 - val_loss: 0.4001
Epoch 3/10
31/31 ━━━━━━━━━━━━━━━━━━━━ 5s 159ms/step - accuracy: 0.8626 - loss: 0.3601 - val_accuracy: 0.8050 - val_loss: 0.5558
Epoch 4/10
31/31 ━━━━━━━━━━━━━━━━━━━━ 5s 157ms/step - accuracy: 0.8928 - loss: 0.3035 - val_accuracy: 0.9419 - val_loss: 0.1922
Epoch 5/10
31/31 ━━━━━━━━━━━━━━━━━━━━ 5s 166ms/step - accuracy: 0.9282 - loss: 0.2116 - val_accuracy: 0.8921 - val_loss: 0.2894
Epoch 6/10
31/31 ━━━━━━━━━━━━━━━━━━━━ 5s 166ms/step - accuracy: 0.9313 - loss: 0.1870 - val_accuracy: 0.9461 - val_loss: 0.1670
Epoch 7/10
31/31 ━━━━━━━━━━━━━━━━━━━━ 5s 163ms/step - accuracy: 0.9407 - loss: 0.1607 - val_accuracy: 0.9419 - val_loss: 0.2221
Epoch 8/10
31/31 ━━━━━━━━━━━━━━━━━━━━ 5s 160ms/step - accuracy: 0.9501 - loss

Model saved as smile_detector_model.h5!


In [4]:
# --- CONFIGURATION ---
IMG_SIZE = (128, 128)          # Must match the size used during training
MODEL_PATH = "smile_detector_model.h5"  # Path to your saved model file

# 1. LOAD THE TRAINED MODEL
print("Loading model...")
model = load_model(MODEL_PATH)
print("Model loaded successfully!")

# 2. DEFINE THE PREDICTION FUNCTION
def predict_smile(image_path):
    """
    Loads a single face image, preprocesses it, and predicts if the person is smiling.
    """
    try:
        # Load and resize the target image
        img = load_img(image_path, target_size=IMG_SIZE)
        
        # Convert image to numpy array and normalize pixel values to [0, 1]
        img_array = img_to_array(img) / 255.0
        
        # Expand dimensions because Keras expects batches: (1, 128, 128, 3)
        img_array = np.expand_dims(img_array, axis=0)
        
        # Get raw probability prediction from the model
        prediction = model.predict(img_array)[0][0]
        
        # Classify based on your mapping: 0 = Smile, 1 = Non-Smile
        if prediction < 0.5:
            confidence = (1 - prediction) * 100
            print(f"\nImage: {image_path}")
            print(f"Result: 😊 SMILING (Confidence: {confidence:.2f}%)")
        else:
            confidence = prediction * 100
            print(f"\nImage: {image_path}")
            print(f"Result: 😐 NOT SMILING (Confidence: {confidence:.2f}%)")
            
    except Exception as e:
        print(f"Error loading image {image_path}: {e}")

# --- 3. CALL THE MODEL ON A NEW IMAGE ---
# Replace 'test_face.jpg' with the actual path to the image you want to check


Loading model...


Model loaded successfully!
Error loading image test_face.jpg: [Errno 2] No such file or directory: 'test_face.jpg'


In [6]:
image_list = [f"test2/{i}.jpg" for i in range(1, 16)]

# Loop through every image file in the array and run the model prediction
print(f"Starting predictions for {len(image_list)} images...\n")

for img_name in image_list:
    import os
    if os.path.exists(img_name):
        predict_smile(img_name)
    else:
        print(f"\nImage: {img_name} -> ❌ File not found in folder.")


Starting predictions for 15 images...

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step

Image: test2/1.jpg
Result: 😊 SMILING (Confidence: 70.50%)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step

Image: test2/2.jpg
Result: 😊 SMILING (Confidence: 89.48%)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step

Image: test2/3.jpg
Result: 😊 SMILING (Confidence: 70.18%)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step

Image: test2/4.jpg
Result: 😊 SMILING (Confidence: 90.38%)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step

Image: test2/5.jpg
Result: 😐 NOT SMILING (Confidence: 98.93%)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step

Image: test2/6.jpg
Result: 😐 NOT SMILING (Confidence: 71.88%)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step

Image: test2/7.jpg
Result: 😊 SMILING (Confidence: 100.00%)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step

Image: test2/8.jpg
Result: 😊 SMILING (Confidence: 98.64%)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step

Image: test2/9.jpg
Result: 😐 NOT SMILING (Confidence: 99.94%)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step

Image: test2/10.jpg
Result: 😊 SMILIN